In [2]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 2 — Per-cohort UMI medians (run after cell 1)
# ──────────────────────────────────────────────────────────────────────────────
COHORT_COL          = "batch"
TOTAL_COUNTS_LAYER  = "counts"
SPLICED_LAYER       = "spliced"
UNSPLICED_LAYER     = "unspliced"

OUTPUT_CSV = Path(
    "/storage/homefs/sv24v923/MPI_data/clean_pipeline/"
    "supp_table_S2_umi_medians.csv"
)

def per_cell_sum(matrix):
    s = matrix.sum(axis=1)
    return np.asarray(s).flatten()

total_per_cell     = per_cell_sum(adata.layers[TOTAL_COUNTS_LAYER])
spliced_per_cell   = per_cell_sum(adata.layers[SPLICED_LAYER])
unspliced_per_cell = per_cell_sum(adata.layers[UNSPLICED_LAYER])

print(f"Total UMI ('{TOTAL_COUNTS_LAYER}'):  range [{total_per_cell.min():.0f}, {total_per_cell.max():.0f}],"
      f"  integer-valued: {np.allclose(total_per_cell, total_per_cell.astype(int))}")
print(f"Spliced ('{SPLICED_LAYER}'):       range [{spliced_per_cell.min():.0f}, {spliced_per_cell.max():.0f}],"
      f"  integer-valued: {np.allclose(spliced_per_cell, spliced_per_cell.astype(int))}")
print(f"Unspliced ('{UNSPLICED_LAYER}'):     range [{unspliced_per_cell.min():.0f}, {unspliced_per_cell.max():.0f}],"
      f"  integer-valued: {np.allclose(unspliced_per_cell, unspliced_per_cell.astype(int))}")
print()

df = adata.obs[[COHORT_COL]].copy()
df["total_umi"]      = total_per_cell
df["spliced_umi"]    = spliced_per_cell
df["unspliced_umi"]  = unspliced_per_cell
# Unspliced fraction defined wrt velocyto-counted reads (spliced + unspliced),
# not the 'counts' layer (different counting rule, includes ambiguous reads).
df["unspliced_frac"] = unspliced_per_cell / np.maximum(spliced_per_cell + unspliced_per_cell, 1)

summary = df.groupby(COHORT_COL).agg(
    n_cells               = ("total_umi",      "size"),
    median_total_UMI      = ("total_umi",      "median"),
    median_spliced_UMI    = ("spliced_umi",    "median"),
    median_unspliced_UMI  = ("unspliced_umi",  "median"),
    median_unspliced_frac = ("unspliced_frac", "median"),
)

# Cast count medians to int for clean display
for col in ["median_total_UMI", "median_spliced_UMI", "median_unspliced_UMI"]:
    summary[col] = summary[col].astype(int)
summary["median_unspliced_frac"] = summary["median_unspliced_frac"].round(3)

# Atlas-wide row
atlas_row = pd.DataFrame({
    "n_cells":               [len(df)],
    "median_total_UMI":      [int(np.median(total_per_cell))],
    "median_spliced_UMI":    [int(np.median(spliced_per_cell))],
    "median_unspliced_UMI":  [int(np.median(unspliced_per_cell))],
    "median_unspliced_frac": [round(float(np.median(df["unspliced_frac"])), 3)],
}, index=["ATLAS_TOTAL"])

summary_with_total = pd.concat([summary, atlas_row])

print("Per-cohort UMI summary:")
print(summary_with_total.to_string())
print()

summary_with_total.to_csv(OUTPUT_CSV)
print(f"Saved → {OUTPUT_CSV}")

Total UMI ('counts'):  range [597, 37933],  integer-valued: True
Spliced ('spliced'):       range [61, 25381],  integer-valued: True
Unspliced ('unspliced'):     range [11, 11712],  integer-valued: True



/scratch/local/3496118/ipykernel_317303/3265483416.py:38: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary = df.groupby(COHORT_COL).agg(


Per-cohort UMI summary:
             n_cells  median_total_UMI  median_spliced_UMI  median_unspliced_UMI  median_unspliced_frac
alsaigh        27813              4496                2695                   633                  0.196
wirka           4230              2717                1737                   347                  0.169
pauli           1328              1547                 814                   333                  0.252
bashore        36246              5787                3201                   955                  0.240
jaiswal         6142              7702                3142                  2228                  0.399
fernandez       3968              2923                1537                   378                  0.194
pan             1906             11211                6050                  2257                  0.275
ATLAS_TOTAL    81633              4869                2780                   725                  0.220

Saved → /storage/homefs/sv24v923/MPI_da